## Unit 07. 非線性方程式之求解 課堂作業
- 課程編號: CHE3502
- 學年度: 114下
- 上課時間: 每週一 16:10-19:20
-
- 指導教授: 莊曜禎 助理教授
- 學生姓名: ＯＯＯ
- 學號: s12345678
- email帳號: fcu.s12345678@gmail.com

---
### 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit07_HW'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit07'
        OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR = OUTPUT_DIR / 'figs'
    else:
        print(f"⚠️ 找不到路徑雲端ChemE-3502路徑，請確認自己的雲端資料夾是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR = OUTPUT_DIR / 'figs'

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar, fsolve, root
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子
np.random.seed(42)

# 設定 matplotlib
plt.rcParams['axes.unicode_minus'] = False

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")

---
## I. 練習題: Van der Waals 方程式求解（單變數非線性方程式）

給定氮氣（N₂）在下列條件下的 Van der Waals 方程式：

$$
f(V) = \left(P + \frac{a}{V^2}\right)(V - b) - RT = 0
$$

**氮氣 Van der Waals 參數**：
- $a = 1.370\ \text{L}^2 \cdot \text{bar/mol}^2$
- $b = 0.0387\ \text{L/mol}$
- $R = 0.08314\ \text{L} \cdot \text{bar/(mol} \cdot \text{K)}$

**操作條件**：$T = 200\ \text{K}$，$P = 50\ \text{bar}$

### **第 1 題：圖形法輔助分析**
- 在 $V \in [0.05,\ 1.0]$ L/mol 範圍內繪製 $f(V)$ 曲線
- 在圖上以水平虛線標示 $y = 0$
- 從圖形中目視判斷方程式有幾個實數根，並估計各根的位置
- 圖表須有適當的英文標題、軸標籤與格線

### **第 2 題：使用 `root_scalar()` 求解**
- 使用 **brentq** 方法，給定包含氣相根的區間，求解氣相摩爾體積 $V_{gas}$
- 使用 **newton** 方法，以理想氣體值 $V_0 = RT/P$ 作為初始猜測值求解
- 比較兩種方法所得的根、迭代次數與函數呼叫次數
- 輸出格式範例：
  ```
  方法: brentq | 根: 0.XXXX L/mol | 函數呼叫次數: XX | 收斂: True
  方法: newton | 根: 0.XXXX L/mol | 函數呼叫次數: XX | 收斂: True
  ```

### **第 3 題：壓力敏感性分析**
- 在 $P \in [10,\ 100]$ bar 範圍內（至少 10 個壓力點），以 brentq 方法逐一求解氣相摩爾體積
- 繪製 $P$ vs $V_{gas}$ 曲線，並與理想氣體 $V_{ideal} = RT/P$ 進行比較
- 計算壓縮因子 $Z = PV/(RT)$，繪製 $P$ vs $Z$ 圖，討論偏差趨勢

In [ ]:
# ============================================================
# 練習題 I 起始程式碼（參數已給定，請在下方完成各題）
# ============================================================

# N2 Van der Waals 參數
a = 1.370      # L^2 bar / mol^2
b = 0.0387     # L/mol
R = 0.08314    # L bar / (mol K)
T = 200.0      # K
P = 50.0       # bar

def vdw_equation(V, T=T, P=P):
    """Van der Waals 方程式：f(V) = 0"""
    return (P + a / V**2) * (V - b) - R * T

# 理想氣體初始猜測值
V_ideal = R * T / P
print(f"理想氣體摩爾體積 V_ideal = {V_ideal:.4f} L/mol")

# ---------- 請在此處完成第 1、2、3 題 ----------


In [ ]:
# ---- 第 1 題：圖形法 ----


In [ ]:
# ---- 第 2 題：使用 root_scalar() 求解 ----


In [ ]:
# ---- 第 3 題：壓力敏感性分析 ----


---
## II. 練習題: 化學平衡系統（多變數聯立非線性方程式）

考慮氣相平衡反應：**乙醇脫氫製乙醛**

$$
\text{C}_2\text{H}_5\text{OH} \rightleftharpoons \text{CH}_3\text{CHO} + \text{H}_2 \quad (Kp_1 = 0.072\ \text{bar})
$$

$$
\text{C}_2\text{H}_5\text{OH} \rightleftharpoons \text{C}_2\text{H}_4 + \text{H}_2\text{O} \quad (Kp_2 = 0.027\ \text{bar})
$$

**初始條件**（純乙醇進料）：
- 初始莫耳數：$n_{EtOH,0} = 1\ \text{mol}$，其餘為 0
- 操作壓力：$P = 1.0\ \text{bar}$，溫度：$T = 500\ \text{K}$
- 設反應進展量：$\xi_1$（第一反應）、$\xi_2$（第二反應）
- 各組分莫耳數：
  - $n_{EtOH} = 1 - \xi_1 - \xi_2$
  - $n_{AcAl} = \xi_1$（乙醛）
  - $n_{H_2} = \xi_1$
  - $n_{Eth} = \xi_2$（乙烯）
  - $n_{H_2O} = \xi_2$
  - $n_{total} = 1 + \xi_1 + \xi_2$

### **第 1 題：建立平衡方程組**
以莫耳分率 $y_i = n_i / n_{total}$ 表示，建立兩個平衡常數方程式：

$$
f_1(\xi_1, \xi_2) = \frac{y_{AcAl} \cdot y_{H_2}}{y_{EtOH}} \cdot P - Kp_1 = 0
$$

$$
f_2(\xi_1, \xi_2) = \frac{y_{Eth} \cdot y_{H_2O}}{y_{EtOH}} \cdot P - Kp_2 = 0
$$

- 撰寫 Python 函式 `equilibrium_equations(xi, Kp1, Kp2, P)` 回傳 `[f1, f2]`

### **第 2 題：使用 `fsolve()` 求解**
- 以 $\xi_1 = 0.05$，$\xi_2 = 0.03$ 作為初始猜測值
- 使用 `scipy.optimize.fsolve()` 求解 $(\xi_1^*,\ \xi_2^*)$
- 輸出平衡時各組分的莫耳分率，並驗證代回結果（殘差）

### **第 3 題：初始猜測值敏感性**
- 嘗試 3 組不同的初始猜測值，分析求解是否收斂至相同結果
- 討論當 $\xi_1$ 或 $\xi_2 < 0$ 時代表什麼物理意義

### **第 4 題：壓力影響分析**
- 在 $P \in [0.5,\ 5.0]$ bar 範圍（10 個壓力點），計算各壓力下的平衡轉化率 $\xi_1^*$ 與 $\xi_2^*$
- 繪製 $P$ vs $\xi_1^*$、$\xi_2^*$ 圖，討論壓力對平衡的影響

In [ ]:
# ============================================================
# 練習題 II 起始程式碼
# ============================================================

# 平衡常數與操作條件
Kp1 = 0.072   # bar  (乙醇 → 乙醛 + H2)
Kp2 = 0.027   # bar  (乙醇 → 乙烯 + H2O)
P   = 1.0     # bar
T   = 500.0   # K

def equilibrium_equations(xi, Kp1=Kp1, Kp2=Kp2, P=P):
    """
    雙反應平衡方程組
    xi = [xi1, xi2]  反應進展量 (mol)
    回傳 [f1, f2]
    """
    xi1, xi2 = xi
    n_total = 1.0 + xi1 + xi2

    # 莫耳分率
    y_EtOH = (1.0 - xi1 - xi2) / n_total
    y_AcAl = xi1 / n_total
    y_H2   = xi1 / n_total
    y_Eth  = xi2 / n_total
    y_H2O  = xi2 / n_total

    f1 = (y_AcAl * y_H2 / y_EtOH) * P - Kp1
    f2 = (y_Eth  * y_H2O / y_EtOH) * P - Kp2
    return [f1, f2]

print("✓ equilibrium_equations 函式定義完成")
print("  物種: EtOH, Acetaldehyde, H2, Ethylene, H2O")

# ---------- 請在此處完成第 1、2、3、4 題 ----------


In [ ]:
# ---- 第 1、2 題：建立方程組 + fsolve 求解 ----


In [ ]:
# ---- 第 3 題：初始猜測值敏感性 ----


In [ ]:
# ---- 第 4 題：壓力影響分析 ----


---
## III. 練習題: 理想溶液泡點計算（Antoine 方程式應用）

對苯（Benzene）/ 甲苯（Toluene）/ 對二甲苯（p-Xylene）三元**理想溶液**，根據 Raoult 定律，泡點條件為：

$$
f(T) = \sum_{i=1}^{3} x_i \cdot P_i^{sat}(T) - P = 0
$$

其中 Antoine 方程式（$\log_{10}(P^{sat}/\text{mmHg}) = A - B/(T + C)$，$T$ 單位為 °C）：

| 物種 | A | B | C |
|------|-------|---------|--------|
| Benzene | 6.90565 | 1211.033 | 220.790 |
| Toluene | 6.95334 | 1343.943 | 219.377 |
| p-Xylene | 6.99052 | 1453.430 | 215.307 |

**操作條件**：$P = 760\ \text{mmHg}$（1 atm），液相組成 $x = [0.3,\ 0.4,\ 0.3]$

### **第 1 題：求解泡點溫度**
- 撰寫 `antoine_psat(T_C, A, B, C)` 函式計算飽和蒸氣壓（單位 mmHg）
- 撰寫 `bubble_point_eq(T_C, x, P)` 函式建立泡點方程
- 使用 `root_scalar()` 的 **brentq** 方法，在 $T \in [60,\ 150]\ °C$ 範圍求解泡點溫度 $T_{bp}$
- 輸出格式：`泡點溫度: XX.XX °C`

### **第 2 題：計算氣相組成**
- 利用 $y_i = x_i \cdot P_i^{sat}(T_{bp}) / P$ 計算氣相組成
- 驗證 $\sum y_i = 1$
- 比較氣液兩相組成，討論輕組分與重組分的分布

### **第 3 題：液相組成影響**
- 固定總壓 $P = 760\ \text{mmHg}$，改變苯的莫耳分率 $x_{Benz} \in [0.1,\ 0.9]$（其餘兩組分等比例分配，即 $x_{Tol} = x_{Xyl} = (1 - x_{Benz})/2$）
- 計算各組成對應的泡點溫度
- 繪製 $x_{Benz}$ vs $T_{bp}$ 曲線，並在圖上同時顯示 $y_{Benz}$（氣相苯的莫耳分率），觀察 $x$-$y$-$T$ 關係

### **第 4 題：壓力影響**
- 固定液相組成 $x = [0.3,\ 0.4,\ 0.3]$，改變操作壓力 $P \in [380,\ 1520]\ \text{mmHg}$（0.5 至 2 atm）
- 計算各壓力下的泡點溫度，觀察壓力對沸點的影響規律

In [ ]:
# ============================================================
# 練習題 III 起始程式碼
# ============================================================

# Antoine 方程式參數 (log10(Psat/mmHg) = A - B/(T+C), T in degC)
antoine_params = {
    'Benzene':  {'A': 6.90565, 'B': 1211.033, 'C': 220.790},
    'Toluene':  {'A': 6.95334, 'B': 1343.943, 'C': 219.377},
    'p-Xylene': {'A': 6.99052, 'B': 1453.430, 'C': 215.307},
}

# 液相組成與操作壓力
x      = np.array([0.3, 0.4, 0.3])  # Benzene, Toluene, p-Xylene
P_mmHg = 760.0                       # mmHg

def antoine_psat(T_C, A, B, C):
    """計算飽和蒸氣壓 (mmHg)，T_C 為攝氏溫度"""
    return 10 ** (A - B / (T_C + C))

def bubble_point_eq(T_C, x, P, params):
    """
    泡點方程 f(T) = sum(xi * Psat_i(T)) - P = 0
    params: list of (A, B, C) for each component
    """
    psat_sum = sum(x[i] * antoine_psat(T_C, **params[i])
                   for i in range(len(x)))
    return psat_sum - P

param_list = [
    antoine_params['Benzene'],
    antoine_params['Toluene'],
    antoine_params['p-Xylene'],
]

print("✓ Antoine 方程式參數設定完成")
print(f"  液相組成 x = {x}")
print(f"  操作壓力 P = {P_mmHg} mmHg ({P_mmHg/760:.1f} atm)")

# ---------- 請在此處完成第 1、2、3、4 題 ----------


In [ ]:
# ---- 第 1、2 題：求解泡點溫度 + 計算氣相組成 ----


In [ ]:
# ---- 第 3 題：液相組成影響 ----


In [ ]:
# ---- 第 4 題：壓力對泡點溫度的影響 ----


---
## 💭 思考題

請在下方文字 cell 中，用自己的話回答以下問題（每題 2-4 句話即可）：

1. **Bisection 法**vs **Newton-Raphson 法**：各自的優缺點是什麼？何時應優先選擇 Bisection？
2. 為什麼多變數非線性方程組無法保證找到**全部解**？`fsolve()` 只能找到幾個解？
3. 在泡點計算中，若 $\sum x_i \cdot P_i^{sat}(T) < P$，代表什麼物理意義？
4. Van der Waals 方程式在某些 $T$、$P$ 條件下可能有**三個根**，分別代表什麼？為何中間根是不穩定的？
5. **收斂失敗**時，有哪些常見的原因和對應的處理策略？

**思考題作答：**

1. （在此輸入你的作答）

2. （在此輸入你的作答）

3. （在此輸入你的作答）

4. （在此輸入你的作答）

5. （在此輸入你的作答）

---
## IV. 額外加分作業（自由繳交）

- 學生可自由選擇是否繳交加分作業（下週上課前完成 email 電子檔）
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分
- 分數加的不多，真的不一定要交加分作業，正常出席上課做好期末報告即可
- 加分作業不一定要全部都做完，想繳交至少完成其中一項實驗即可
- 請務必自行完成！你可以問 AI、問同學、問學長姊，但一定要自行**吸收、執行、整理**結果！
- 不要貼 AI 的答案寄給老師看，你自己看就好
- 如果系統自動比對發現作業內容（與他人雷同、原創性低、抄襲比例過高），作業分數會 0 分
- 老師看你作業要花時間，真的有心想寫加分作業，請好好自己做

---
### **實驗1：CSTR 多重穩態分析**

以下列 CSTR 穩態方程組（物料平衡 + 能量平衡）為對象：

$$
f_1(x_1, x_2) = -x_1 + Da (1 - x_1) e^{x_2/(1 + x_2/\gamma)} = 0
$$
$$
f_2(x_1, x_2) = -x_2 + B \cdot Da (1 - x_1) e^{x_2/(1 + x_2/\gamma)} - \beta (x_2 - x_{2c}) = 0
$$

其中 $Da = 0.072$，$B = 8$，$\gamma = 20$，$\beta = 0.3$，$x_{2c} = 0$。$x_1$ 為無因次濃度，$x_2$ 為無因次溫度。

**要求**：
- 使用多起始點搜尋法（系統化掃描 $x_1 \in [0,1]$、$x_2 \in [0,6]$）找出**所有穩態**
- 繪製相圖（$f_1 = 0$ 與 $f_2 = 0$ 的零等值線），標示所有穩態點
- 討論各穩態的物理意義（低、中、高轉化率穩態）

---
### **實驗2：SRK 狀態方程式與 VdW 比較**

使用 Soave-Redlich-Kwong（SRK）方程式求解 CO₂（T=300 K, P=50 bar）的摩爾體積：

$$
P = \frac{RT}{V - b} - \frac{a \cdot \alpha(T)}{V(V+b)}
$$

其中 $a = 0.42748 R^2 T_c^2/P_c$，$b = 0.08664 RT_c/P_c$，
$\alpha(T) = [1 + m(1 - \sqrt{T_r})]^2$，$m = 0.48 + 1.574\omega - 0.176\omega^2$

**CO₂ 臨界參數**：$T_c = 304.2\ \text{K}$，$P_c = 73.8\ \text{bar}$，$\omega = 0.225$

**要求**：
- 使用 `root_scalar()` 求解 SRK 方程式，求得 CO₂ 氣相摩爾體積 $V_{SRK}$
- 與 VdW 方程式（$a = 3.658\ \text{L}^2·\text{bar/mol}^2$，$b = 0.04286\ \text{L/mol}$）的結果比較
- 計算各方程式的壓縮因子 $Z$，討論哪個方程式對 CO₂ 的描述更準確

---
### **實驗3：多起始點法的效能分析**

以練習題 I 的 VdW 方程式（N₂, 150K, 80 bar）在更高壓力可能出現多根的條件下：

**要求**：
- 設計一個系統化的掃描程式，在 $V \in [0.05, 2.0]$ L/mol 範圍搜尋所有符號變化區間
- 對每個搜尋到的根，以 `brentq` 精確求解
- 計算每次求解的函數呼叫次數，討論掃描粒度（節點數量）對「找到所有根」的可靠性影響

In [ ]:
# ---- 加分作業：請在此處作答（選擇實驗1、2、3 其中一項或多項完成）----


---
# 想對老師說的話